# 01 · Ingesta

Descarga las cuatro fuentes del proyecto a `data/raw/` y las lee con PySpark para verificar esquema, volumen y cobertura temporal.

| Fuente | Institución | Formato | Periodicidad |
| :--- | :--- | :--- | :--- |
| Ingresos y gastos del Gobierno Federal | SHCP | CSV | Mensual |
| Ingresos tributarios | SAT | XLS → CSV | Mensual |
| IGAE (base 2018) | INEGI | CSV (zip) | Mensual |
| INPC | INEGI | CSV (zip) | Mensual |

Ejecutar desde `notebooks/`. Los datos no se versionan (`data/` está en `.gitignore`); este notebook es la forma de obtenerlos. Detalle de cada fuente en [docs/fuentes.md](../docs/fuentes.md).

In [12]:
# ============================================================
# 1. LIBRERÍAS Y RUTAS
# ============================================================

import shutil
import ssl
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

PERIODO = (201501, 202503)  # AAAAMM; termina en 2025-03 porque el IGAE de datos abiertos no tiene meses posteriores

print("Datos crudos en:", RAW)

Datos crudos en: I:\Mi unidad\1ro. MCD\III. Procesamiento de grandes bases de datos\Integrador-PGBD-26A\data\raw


In [13]:
# ============================================================
# 2. FUENTES
# ============================================================

FUENTES = {
    "ingreso_gasto_finan.csv": "https://www.secciones.hacienda.gob.mx/work/models/estadisticas_oportunas/datos_abiertos_eopf/ingreso_gasto_finan.csv",
    "IngresosTributarios.xls": "https://wu1agsprosta001.blob.core.windows.net/agsc-publicaciones/Datos_abiertos/Cifras_SAT/Documents/IngresosTributarios.xls",
    "igae_mensual_csv.zip": "https://www.inegi.org.mx/contenidos/programas/igae/2018/datosabiertos/igae_mensual_csv.zip",
    "inpc_mensual_csv.zip": "https://www.inegi.org.mx/contenidos/programas/inpc/2018a/datosabiertos/conjunto_de_datos_inpc_indicador_mensual_csv.zip",
}

In [14]:
# ============================================================
# 3. DESCARGA
# ============================================================
#
# El servidor de la SHCP no envía el certificado intermedio
# correcto (Let's Encrypt YR1). En lugar de desactivar la
# verificación SSL, descargamos ese intermedio y lo agregamos
# al contexto: la conexión sigue verificada contra la raíz.
#
# Si el archivo ya existe no se vuelve a descargar.
# Para actualizar una fuente, borrar su archivo de data/raw/.
# ============================================================

ctx = ssl.create_default_context()
yr1 = urllib.request.urlopen("http://yr1.i.lencr.org/", timeout=60).read()
ctx.load_verify_locations(cadata=ssl.DER_cert_to_PEM_cert(yr1))

for nombre, url in FUENTES.items():
    destino = RAW / nombre
    if not destino.exists():
        tmp = destino.with_suffix(".part")
        with urllib.request.urlopen(url, context=ctx, timeout=600) as r, open(tmp, "wb") as f:
            shutil.copyfileobj(r, f)
        tmp.rename(destino)  # evita dejar archivos a medias si falla la descarga
    print(f"{nombre:28} {destino.stat().st_size / 1e6:8.1f} MB")

ingreso_gasto_finan.csv         224.9 MB
IngresosTributarios.xls           0.2 MB
igae_mensual_csv.zip              0.4 MB
inpc_mensual_csv.zip              0.0 MB


In [15]:
# ============================================================
# 4. DESCOMPRESIÓN (INEGI)
# ============================================================

for zip_name in ["igae_mensual_csv.zip", "inpc_mensual_csv.zip"]:
    with zipfile.ZipFile(RAW / zip_name) as z:
        z.extractall(RAW / zip_name.removesuffix(".zip"))

RUTA_IGAE = next((RAW / "igae_mensual_csv" / "conjunto_de_datos").glob("conjunto_de_datos_igae_*.csv"))
RUTA_INPC = RAW / "inpc_mensual_csv" / "conjunto_de_datos" / "conjunto_de_datos_inpc_mensual.csv"
print(RUTA_IGAE.name)
print(RUTA_INPC.name)

conjunto_de_datos_igae_igae2025_03.csv
conjunto_de_datos_inpc_mensual.csv


In [16]:
# ============================================================
# 5. SAT: XLS → CSV
# ============================================================
#
# Spark no lee Excel de forma nativa. El archivo tiene:
# - título en las filas 1-4;
# - encabezado en la fila 5 (y subencabezados de IEPS en la 6);
# - notas al pie al final.
#
# Solo cambiamos de formato y quitamos notas; la limpieza
# (meses en texto, cifras preliminares) va en transformaciones.
# ============================================================

sat_xls = pd.read_excel(RAW / "IngresosTributarios.xls", header=None).dropna(axis=1, how="all")

encabezado = sat_xls.iloc[5].fillna(sat_xls.iloc[4]).str.strip()
sat = sat_xls.iloc[6:].set_axis(encabezado, axis=1)
sat = sat[sat["Mes"].notna()]  # las notas al pie no tienen mes

RUTA_SAT = RAW / "ingresos_tributarios_sat.csv"
sat.to_csv(RUTA_SAT, index=False)

print(sat.shape)
sat.head()

(201, 22)


5,Año,Mes,Total de Ingresos tributarios,Impuesto Sobre la Renta,Impuesto al Valor Agregado,Impuesto Especial sobre Producción y Servicios,Gasolinas y diésel,Tabacos labrados,Alcohol,Cerveza y bebidas refrescantes,...,Bebidas energetizantes,Bebidas saborizadas,Alimentos no básicos con alta densidad calórica,Plaguicidas,Carbono,Importaciones,Exportaciones,Impuesto sobre Automóviles Nuevos,Impuesto por la Actividad de Exploración y Extracción de Hidrocarburos,Otros ingresos tributarios
6,2010,Enero,143806.723,71997.945,58816.947,2628.27,-6928.45,6808.447,734.741,1865.145,...,0,n.d.,n.d.,n.d.,n.d.,1988.947,0.004,559.868,n.d.,7814.742
7,2010,Febrero,99163.694,46098.298,36601.913,-1320.321,-4171.001,655.931,288.865,1343.903,...,0,n.d.,n.d.,n.d.,n.d.,1814.108,0.017,333.424,n.d.,15636.255
8,2010,Marzo,110803.305,55628.337,35756.921,-2675.345,-5954.753,339.342,843.49,1455.825,...,0,n.d.,n.d.,n.d.,n.d.,2111.401,0.064,349.452,n.d.,19632.475
9,2010,Abril,105036.462,53859.74,40199.949,-981.862,-5423.797,1621.642,301.466,1684.927,...,0,n.d.,n.d.,n.d.,n.d.,1767.535,0.008,389.153,n.d.,9801.939
10,2010,Mayo,90663.865,45604.199,37907.734,-1953.271,-6539.372,1656.855,352.475,1810.576,...,0,n.d.,n.d.,n.d.,n.d.,1991.99,0.031,335.969,n.d.,6777.213


In [17]:
# ============================================================
# 6. SPARK SESSION
# ============================================================

spark = (
    SparkSession.builder
    .appName("Integrador_01_Ingesta")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark

In [18]:
# ============================================================
# 7. LECTURA CON SPARK
# ============================================================
#
# inferSchema por ahora: es exploración. Cuando el pipeline
# esté estable conviene declarar el esquema explícito
# (ver docs/guia_optimizacion.md).
# ============================================================

leer = lambda ruta: spark.read.csv(str(ruta), header=True, inferSchema=True)

dfs = {
    "shcp": leer(RAW / "ingreso_gasto_finan.csv"),
    "sat": leer(RUTA_SAT),
    "igae": leer(RUTA_IGAE),
    "inpc": leer(RUTA_INPC),
}

for nombre, df in dfs.items():
    print(f"{nombre:5} filas={df.count():>9,}  columnas={len(df.columns)}")

shcp  filas=  746,676  columnas=16
sat   filas=      201  columnas=22
igae  filas=      120  columnas=430
inpc  filas=    2,002  columnas=7


In [19]:
dfs["shcp"].printSchema()
dfs["inpc"].printSchema()

root
 |-- CICLO: integer (nullable = true)
 |-- MES: string (nullable = true)
 |-- CLAVE_DE_CONCEPTO: string (nullable = true)
 |-- NOMBRE: string (nullable = true)
 |-- TEMA: string (nullable = true)
 |-- SUBTEMA: string (nullable = true)
 |-- SECTOR: string (nullable = true)
 |-- AMBITO: string (nullable = true)
 |-- TIPO_DE_INFORMACION: string (nullable = true)
 |-- BASE_DE_REGISTRO: string (nullable = true)
 |-- UNIDAD_DE_MEDIDA: string (nullable = true)
 |-- PERIODO_INICIO: timestamp (nullable = true)
 |-- PERIODO_FINAL: timestamp (nullable = true)
 |-- FRECUENCIA: string (nullable = true)
 |-- DIFUSION: string (nullable = true)
 |-- MONTO: double (nullable = true)

root
 |-- COBERTURA: string (nullable = true)
 |-- PERIODICIDAD: string (nullable = true)
 |-- FECHA: string (nullable = true)
 |-- CONCEPTO: string (nullable = true)
 |-- VALOR: double (nullable = true)
 |-- UNIDAD_MEDIDA: string (nullable = true)
 |-- ESTATUS: string (nullable = true)



In [21]:
# ============================================================
# 8. COBERTURA TEMPORAL
# ============================================================

MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio",
         "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
mes_num = F.create_map(*[x for i, m in enumerate(MESES, 1) for x in (F.lit(m), F.lit(i))])
mes_abrev_num = F.create_map(*[x for i, m in enumerate(MESES, 1) for x in (F.lit(m[:3]), F.lit(i))])

def rango(df, anio, mes):
    # (min, max) como AAAAMM. Falla si algún mes no se pudo interpretar.
    r = df.select((anio * 100 + mes).alias("p")).agg(F.min("p"), F.max("p"), F.count_if(F.col("p").isNull())).first()
    assert r[2] == 0, f"{r[2]} filas con año/mes no reconocido"
    return r[0], r[1]

sat_anio = F.regexp_extract(F.col("Año").cast("string"), r"(\d{4})", 1).cast("int")  # algunos años traen "*" (preliminar)

inpc = dfs["inpc"].where(F.col("CONCEPTO").endswith("Precios al Consumidor (INPC)"))
inpc.select("FECHA").show(5, truncate=False)  # muestra el formato real en consola

f_str = F.trim(F.col("FECHA"))
inpc_anio = F.coalesce(
    F.regexp_extract(f_str, r"^(\d{4})[-/]\d{1,2}", 1).cast("int"),             # AAAA/MM o AAAA-MM-DD
    F.regexp_extract(f_str, r"^\d{1,2}[-/]\d{1,2}[-/](\d{4})$", 1).cast("int"), # DD/MM/AAAA
    F.regexp_extract(f_str, r"^\d{1,2}[-/](\d{4})$", 1).cast("int"),            # MM/AAAA
    F.regexp_extract(f_str, r"(\d{4})", 1).cast("int")                          # "Ene 2015" o AAAAMM
)
inpc_mes = F.coalesce(
    F.regexp_extract(f_str, r"^\d{4}[-/](\d{1,2})", 1).cast("int"),             # AAAA/MM o AAAA-MM-DD
    F.regexp_extract(f_str, r"^\d{1,2}[-/](\d{1,2})[-/]\d{4}$", 1).cast("int"), # DD/MM/AAAA
    F.regexp_extract(f_str, r"^(\d{1,2})[-/]\d{4}$", 1).cast("int"),            # MM/AAAA
    F.regexp_extract(f_str, r"^\d{4}(\d{2})$", 1).cast("int"),                  # AAAAMM
    mes_abrev_num[F.initcap(F.regexp_extract(f_str, r"^([A-Za-z]{3})", 1))]     # "Ene 2015"
)

# IGAE: las columnas llegan a diciembre aunque estén vacías; tomamos los meses con dato
# de la serie total. Los encabezados traen marcas <R> (revisado) y <P> (preliminar).
igae_total = dfs["igae"].where(F.col("Descriptores") == "Índice de volumen físico base 2018=100|Total").first().asDict()
igae_p = [int(c[:4]) * 100 + MESES.index(c[5:].split("<")[0]) + 1
          for c, v in igae_total.items() if "|" in c and "Anual" not in c and v is not None]

rangos = {
    "shcp": rango(dfs["shcp"], F.col("CICLO"), mes_num[F.col("MES")]),
    "sat":  rango(dfs["sat"], sat_anio, mes_num[F.col("Mes")]),
    "inpc": rango(inpc, inpc_anio, inpc_mes),
    "igae": (min(igae_p), max(igae_p)),
}

for nombre, (ini, fin) in rangos.items():
    ok = ini <= PERIODO[0] and fin >= PERIODO[1]
    print(f"{nombre:5} {ini} a {fin}  {'OK' if ok else 'NO CUBRE'}")
    assert ok, f"{nombre} no cubre el periodo {PERIODO}"

+----------+
|FECHA     |
+----------+
|01/01/2003|
|01/02/2003|
|01/03/2003|
|01/04/2003|
|01/05/2003|
+----------+
only showing top 5 rows

shcp  201101 a 202609  OK
sat   201001 a 202609  OK
inpc  200301 a 202610  OK
igae  199301 a 202503  OK


### Estructura final del repositorio

```text
Integrador-PGBD-26A/
├── data/
│   ├── raw/                        # Generado por notebooks/01_ingesta.ipynb (en .gitignore)
│   └── processed/                  # Salida analítica en Parquet particionado por año
├── docs/                           # Documentación técnica (fuentes.md, arquitectura, etc.)
├── notebooks/
│   └── 01_ingesta.ipynb            # Descarga, descompresión y conversión SAT XLS -> CSV
├── reports/                        # Evidencia de ejecución (conteos, validaciones y esquema final)
├── src/
│   ├── __init__.py
│   ├── pipeline.py                 # Orquestador ejecutable de principio a fin (python -m src.pipeline)
│   ├── ingestion/
│   │   ├── __init__.py
│   │   └── loader.py               # Etapa 1: Extracción con esquemas explícitos y rutas
│   ├── quality/
│   │   ├── __init__.py
│   │   └── validators.py           # Etapa 3 y 5: Validaciones pre/post limpieza y pre-carga
│   └── transformations/
│       ├── __init__.py
│       ├── clean.py                # Etapa 2: Limpieza, llave `fecha` (date) y homologación a millones
│       └── integrate.py            # Etapa 4 y 5: Pivot SHCP, join broadcast, ventanas y Parquet
├── .gitignore
├── README.md
└── requirements.txt